In [1]:
import pandas as pd
from pathlib import Path

project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

data_path = project_path / "data" / "processed"

train_df = pd.read_csv(data_path / "train.csv")
val_df = pd.read_csv(data_path / "validation.csv")

X_train = train_df["text"]
y_train = train_df["label"]

X_val = val_df["text"]
y_val = val_df["label"]

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Classes:", y_train.nunique())

Training: 803
Validation: 178
Classes: 24


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True)),
    ("classifier", LinearSVC(random_state=42))
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "classifier__C": [0.1, 1, 10]
}

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True
)

grid_search.fit(
    X_train,
    y_train,
    groups=train_df["group_id"]
)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest CV Macro F1:",
      round(grid_search.best_score_, 4))

Best parameters:
{'classifier__C': 10, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}

Best CV Macro F1: 0.9578


In [3]:
from sklearn.metrics import accuracy_score, f1_score

best_svm = grid_search.best_estimator_

predictions_tuned = best_svm.predict(X_val)

print("Tuned SVM - Validation Results")

print("Accuracy:", round(
    accuracy_score(y_val, predictions_tuned), 4
))

print("Macro F1:", round(
    f1_score(y_val, predictions_tuned, average="macro"), 4
))

Tuned SVM - Validation Results
Accuracy: 0.9944
Macro F1: 0.9952


In [4]:
comparison = pd.DataFrame([
    {
        "Model": "Baseline SVM",
        "Accuracy": 0.9775,
        "Macro F1": 0.9793
    },
    {
        "Model": "Tuned SVM",
        "Accuracy": accuracy_score(
            y_val, predictions_tuned
        ),
        "Macro F1": f1_score(
            y_val,
            predictions_tuned,
            average="macro"
        )
    }
])

display(comparison.round(4))

,Model,Accuracy,Macro F1
0,Baseline SVM,0.9775,0.9793
1,Tuned SVM,0.9944,0.9952


In [5]:
errors = pd.DataFrame({
    "text": X_val.values,
    "actual_disease": y_val.values,
    "predicted_disease": predictions_tuned
})

errors = errors[
    errors["actual_disease"]
    != errors["predicted_disease"]
]

print("Total incorrect predictions:", len(errors))

display(errors)

Total incorrect predictions: 1


,text,actual_disease,predicted_disease
21,I'm feeling fatigued and have no energy. I can...,Chicken pox,Common Cold


In [6]:
import json
import joblib

models_path = project_path / "models"
models_path.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_svm,
    models_path / "svm_tuned_validation.joblib"
)

tuning_info = {
    "model": "LinearSVC",
    "best_parameters": grid_search.best_params_,
    "cv_macro_f1": float(grid_search.best_score_),
    "validation_accuracy": float(
        accuracy_score(y_val, predictions_tuned)
    ),
    "validation_macro_f1": float(
        f1_score(y_val, predictions_tuned, average="macro")
    )
}

with open(models_path / "svm_tuning_results.json", "w") as f:
    json.dump(tuning_info, f, indent=4)

print("Tuned SVM and results saved successfully!")

Tuned SVM and results saved successfully!


In [7]:
for _, row in errors.iterrows():
    print("Actual disease:", row["actual_disease"])
    print("Predicted disease:", row["predicted_disease"])
    print("\nFull symptom description:")
    print(row["text"])

Actual disease: Chicken pox
Predicted disease: Common Cold

Full symptom description:
I'm feeling fatigued and have no energy. I can barely keep my eyes open during the day, and I've been feeling lethargic and unable to motivate myself.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

pipeline_lr = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

param_grid_lr = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "classifier__C": [0.1, 1, 10]
}

cv_lr = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    scoring="f1_macro",
    cv=cv_lr,
    n_jobs=-1,
    refit=True
)

grid_lr.fit(
    X_train,
    y_train,
    groups=train_df["group_id"]
)

print("Best LR Parameters:")
print(grid_lr.best_params_)

print("\nBest CV Macro F1:",
      round(grid_lr.best_score_, 4))

Best LR Parameters:
{'classifier__C': 10, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1)}

Best CV Macro F1: 0.9554


In [9]:
from sklearn.metrics import accuracy_score, f1_score

best_lr = grid_lr.best_estimator_

predictions_lr_tuned = best_lr.predict(X_val)

lr_accuracy = accuracy_score(
    y_val, predictions_lr_tuned
)

lr_f1 = f1_score(
    y_val,
    predictions_lr_tuned,
    average="macro"
)

print("Tuned Logistic Regression Results")
print("Accuracy:", round(lr_accuracy, 4))
print("Macro F1:", round(lr_f1, 4))

Tuned Logistic Regression Results
Accuracy: 0.9775
Macro F1: 0.9787


In [10]:
comparison_lr = pd.DataFrame([
    {
        "Model": "Baseline Logistic Regression",
        "Accuracy": 0.9551,
        "Macro F1": 0.9557
    },
    {
        "Model": "Tuned Logistic Regression",
        "Accuracy": lr_accuracy,
        "Macro F1": lr_f1
    }
])

display(comparison_lr.round(4))

,Model,Accuracy,Macro F1
0,Baseline Logistic Regression,0.9551,0.9557
1,Tuned Logistic Regression,0.9775,0.9787


In [11]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

# Krijimi i Pipeline per KNN
pipeline_knn = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True)),
    ("classifier", KNeighborsClassifier(
        metric="cosine"
    ))
])

# Parametrat qe do te testojme
param_grid_knn = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "classifier__n_neighbors": [1, 3, 5, 7],
    "classifier__weights": ["uniform", "distance"]
}

# Cross-Validation
cv_knn = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_knn = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=param_grid_knn,
    scoring="f1_macro",
    cv=cv_knn,
    n_jobs=-1,
    refit=True
)

# Training me grupet ekzistuese
grid_knn.fit(
    X_train,
    y_train,
    groups=train_df["group_id"]
)

print("Best KNN Parameters:")
print(grid_knn.best_params_)

print("\nBest CV Macro F1:",
      round(grid_knn.best_score_, 4))

Best KNN Parameters:
{'classifier__n_neighbors': 1, 'classifier__weights': 'uniform', 'tfidf__ngram_range': (1, 1)}

Best CV Macro F1: 0.9299


In [12]:
from sklearn.metrics import accuracy_score, f1_score

best_knn = grid_knn.best_estimator_

predictions_knn_tuned = best_knn.predict(X_val)

knn_accuracy = accuracy_score(
    y_val, predictions_knn_tuned
)

knn_f1 = f1_score(
    y_val,
    predictions_knn_tuned,
    average="macro"
)

print("Tuned KNN Results")
print("Accuracy:", round(knn_accuracy, 4))
print("Macro F1:", round(knn_f1, 4))

Tuned KNN Results
Accuracy: 0.9775
Macro F1: 0.9777


In [13]:
comparison_knn = pd.DataFrame([
    {
        "Model": "Baseline KNN",
        "Accuracy": 0.9045,
        "Macro F1": 0.9078
    },
    {
        "Model": "Tuned KNN",
        "Accuracy": knn_accuracy,
        "Macro F1": knn_f1
    }
])

display(comparison_knn.round(4))

,Model,Accuracy,Macro F1
0,Baseline KNN,0.9045,0.9078
1,Tuned KNN,0.9775,0.9777


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

# 1. Krijimi i Pipeline
pipeline_rf = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True)),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ))
])

# 2. Parametrat per Hyperparameter Tuning
param_grid_rf = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 20],
    "classifier__min_samples_leaf": [1, 2]
}

# 3. Cross-Validation
cv_rf = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# 4. Krijimi i GridSearchCV
grid_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    scoring="f1_macro",
    cv=cv_rf,
    n_jobs=-1,
    refit=True
)

# 5. Trajnimi
grid_rf.fit(
    X_train,
    y_train,
    groups=train_df["group_id"]
)

# 6. Shfaqja e rezultateve
print("Best Random Forest Parameters:")
print(grid_rf.best_params_)

print("\nBest CV Macro F1:",
      round(grid_rf.best_score_, 4))

Best Random Forest Parameters:
{'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 200, 'tfidf__ngram_range': (1, 2)}

Best CV Macro F1: 0.9173


In [16]:
from sklearn.metrics import accuracy_score, f1_score

best_rf = grid_rf.best_estimator_

predictions_rf_tuned = best_rf.predict(X_val)

rf_accuracy = accuracy_score(y_val, predictions_rf_tuned)

rf_f1 = f1_score(
    y_val,
    predictions_rf_tuned,
    average="macro"
)

print("Tuned Random Forest Results")
print("Accuracy:", round(rf_accuracy, 4))
print("Macro F1:", round(rf_f1, 4))

Tuned Random Forest Results
Accuracy: 0.9382
Macro F1: 0.9386


In [2]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

pipeline_mlp = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        max_features=2000
    )),
    ("classifier", MLPClassifier(
        activation="relu",
        solver="adam",
        max_iter=200,
        random_state=42
    ))
])

param_grid_mlp = {
    "classifier__hidden_layer_sizes": [(32,), (64,)],
    "classifier__alpha": [0.0001, 0.01],
    "tfidf__ngram_range": [(1, 1), (1, 2)]
}

cv_mlp = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_mlp = GridSearchCV(
    estimator=pipeline_mlp,
    param_grid=param_grid_mlp,
    scoring="f1_macro",
    cv=cv_mlp,
    n_jobs=1,
    refit=True
)

grid_mlp.fit(
    X_train,
    y_train,
    groups=train_df["group_id"]
)

print("Best MLP Parameters:")
print(grid_mlp.best_params_)

print("\nBest CV Macro F1:",
      round(grid_mlp.best_score_, 4))

c:\Projects\Bachelor-Thesis\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Projects\Bachelor-Thesis\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Projects\Bachelor-Thesis\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Projects\Bachelor-Thesis\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Projects\Bachelor-Thesis\.venv\Li

Best MLP Parameters:
{'classifier__alpha': 0.01, 'classifier__hidden_layer_sizes': (64,), 'tfidf__ngram_range': (1, 1)}

Best CV Macro F1: 0.9499


c:\Projects\Bachelor-Thesis\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [3]:
from sklearn.metrics import accuracy_score, f1_score

best_mlp = grid_mlp.best_estimator_

predictions_mlp_tuned = best_mlp.predict(X_val)

mlp_accuracy = accuracy_score(
    y_val, predictions_mlp_tuned
)

mlp_f1 = f1_score(
    y_val,
    predictions_mlp_tuned,
    average="macro"
)

print("Tuned MLP Results")
print("Accuracy:", round(mlp_accuracy, 4))
print("Macro F1:", round(mlp_f1, 4))

Tuned MLP Results
Accuracy: 0.9775
Macro F1: 0.9772


In [4]:
comparison_mlp = pd.DataFrame([
    {
        "Model": "Baseline MLP",
        "Accuracy": 0.9551,
        "Macro F1": 0.9593
    },
    {
        "Model": "Tuned MLP",
        "Accuracy": mlp_accuracy,
        "Macro F1": mlp_f1
    }
])

display(comparison_mlp.round(4))

,Model,Accuracy,Macro F1
0,Baseline MLP,0.9551,0.9593
1,Tuned MLP,0.9775,0.9772


In [5]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score

# Kopjojme konfigurimin me te mire te MLP
mlp_400 = clone(best_mlp)

# Rrisim kufirin e iteracioneve
mlp_400.set_params(classifier__max_iter=400)

# Trajnojme perseri vetem kete konfigurim
mlp_400.fit(X_train, y_train)

# Vleresojme ne Validation
predictions_mlp_400 = mlp_400.predict(X_val)

print("MLP with max_iter=400")
print("Training iterations:",
      mlp_400.named_steps["classifier"].n_iter_)

print("Accuracy:", round(
    accuracy_score(y_val, predictions_mlp_400), 4
))

print("Macro F1:", round(
    f1_score(y_val, predictions_mlp_400, average="macro"), 4
))

MLP with max_iter=400
Training iterations: 373
Accuracy: 0.9775
Macro F1: 0.979


In [7]:
import pandas as pd
from pathlib import Path

project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

models_path = project_path / "models"
models_path.mkdir(parents=True, exist_ok=True)

# Rezultatet e Validation nga eksperimentet e kryera
tuned_results = pd.DataFrame([
    {
        "Model": "SVM",
        "Accuracy": 0.9944,
        "Macro F1": 0.9952
    },
    {
        "Model": "Logistic Regression",
        "Accuracy": 0.9775,
        "Macro F1": 0.9787
    },
    {
        "Model": "KNN",
        "Accuracy": 0.9775,
        "Macro F1": 0.9777
    },
    {
        "Model": "Random Forest",
        "Accuracy": 0.9382,
        "Macro F1": 0.9386
    },
    {
        "Model": "Neural Network (MLP)",
        "Accuracy": 0.9775,
        "Macro F1": 0.9790
    }
])

tuned_results = tuned_results.sort_values(
    "Macro F1",
    ascending=False
)

tuned_results.to_csv(
    models_path / "tuned_results.csv",
    index=False
)

print("Tuned results saved successfully!")
display(tuned_results)

Tuned results saved successfully!


,Model,Accuracy,Macro F1
0,SVM,0.9944,0.9952
4,Neural Network (MLP),0.9775,0.9790
1,Logistic Regression,0.9775,0.9787
2,KNN,0.9775,0.9777
3,Random Forest,0.9382,0.9386


In [8]:
print("MLP exists:", "mlp_400" in globals())

MLP exists: True


In [9]:
import joblib

joblib.dump(
    mlp_400,
    models_path / "mlp_tuned_validation.joblib"
)

print("MLP model saved successfully!")

MLP model saved successfully!
